In [2]:
import re
from datetime import datetime, timedelta

class AgenteTareasCurso:
    def __init__(self):
        self.tareas = []  # lista de diccionarios: nombre, fecha_limite, prioridad, completada
        self.base_conocimiento = {
            "agregar": "Para agregar una tarea, dime: 'agregar tarea <nombre> para el <YYYY-MM-DD> con prioridad <alta/media/baja>'",
            "listar": "Usa 'listar tareas' para ver todas tus tareas.",
            "completar": "Usa 'completar tarea <nombre>' para marcar como hecha.",
            "eliminar": "Usa 'eliminar tarea <nombre>' para borrarla.",
            "recordatorios": "Te recordaré las tareas próximas (vencen en menos de 2 días)"
        }

    # --- Inteligencia para entender la pregunta ---
    def clasificar_intencion(self, texto):
        texto = texto.lower()
        if re.search(r"agregar|nueva tarea|añadir", texto):
            return "agregar"
        elif re.search(r"listar|muestra|qué tareas|pendientes", texto):
            return "listar"
        elif re.search(r"completar|hecho|realizada|finalizar", texto):
            return "completar"
        elif re.search(r"eliminar|borrar|quitar", texto):
            return "eliminar"
        elif re.search(r"recordatorio|próximo|vencer", texto):
            return "recordatorio"
        else:
            return "ayuda"

    # --- Extraer parámetros de la frase ---
    def extraer_nombre(self, texto):
        # Busca después de "tarea" o "agregar"
        match = re.search(r"(?:tarea|agregar)\s+([a-zA-Záéíóúñ\s]+)(?:para|con|$)", texto.lower())
        if match:
            return match.group(1).strip()
        # si no, toma primeras palabras hasta encontrar fecha o prioridad
        palabras = texto.split()
        for i, p in enumerate(palabras):
            if p in ["para", "con", "prioridad", "fecha"]:
                return " ".join(palabras[1:i]) if i>1 else "tarea"
        return "tarea_sin_nombre"

    def extraer_fecha(self, texto):
        match = re.search(r"(\d{4}-\d{2}-\d{2})", texto)
        if match:
            return match.group(1)
        # fechas relativas: mañana, hoy
        if "mañana" in texto:
            return (datetime.now() + timedelta(days=1)).strftime("%Y-%m-%d")
        if "pasado mañana" in texto:
            return (datetime.now() + timedelta(days=2)).strftime("%Y-%m-%d")
        return (datetime.now() + timedelta(days=7)).strftime("%Y-%m-%d")  # por defecto una semana

    def extraer_prioridad(self, texto):
        if "alta" in texto:
            return 1
        elif "media" in texto:
            return 2
        elif "baja" in texto:
            return 3
        return 2  # prioridad media por defecto

    # --- Acciones ---
    def agregar_tarea(self, comando):
        nombre = self.extraer_nombre(comando)
        fecha = self.extraer_fecha(comando)
        prioridad = self.extraer_prioridad(comando)
        self.tareas.append({
            "nombre": nombre,
            "fecha_limite": fecha,
            "prioridad": prioridad,
            "completada": False
        })
        return f"✅ Tarea '{nombre}' agregada para el {fecha} (prioridad {prioridad})."

    def listar_tareas(self):
        if not self.tareas:
            return "📭 No hay tareas pendientes."
        # ordenar por fecha y prioridad
        pendientes = [t for t in self.tareas if not t["completada"]]
        pendientes.sort(key=lambda x: (x["fecha_limite"], x["prioridad"]))
        resultado = "📋 Tareas pendientes:\n"
        for i, t in enumerate(pendientes, 1):
            estado = "🔴" if t["fecha_limite"] < datetime.now().strftime("%Y-%m-%d") else "🟢"
            prioridad_str = {1:"Alta",2:"Media",3:"Baja"}[t["prioridad"]]
            resultado += f"{i}. {estado} {t['nombre']} - límite: {t['fecha_limite']} (prioridad {prioridad_str})\n"
        return resultado

    def completar_tarea(self, comando):
        nombre = self.extraer_nombre(comando)
        for t in self.tareas:
            if t["nombre"] == nombre and not t["completada"]:
                t["completada"] = True
                return f"🎉 Tarea '{nombre}' marcada como completada."
        return f"❌ No se encontró la tarea pendiente '{nombre}'."

    def eliminar_tarea(self, comando):
        nombre = self.extraer_nombre(comando)
        original_len = len(self.tareas)
        self.tareas = [t for t in self.tareas if t["nombre"] != nombre]
        if len(self.tareas) < original_len:
            return f"🗑️ Tarea '{nombre}' eliminada."
        return f"❌ No se encontró la tarea '{nombre}'."

    def recordatorios(self):
        hoy = datetime.now().strftime("%Y-%m-%d")
        proximas = []
        for t in self.tareas:
            if not t["completada"] and t["fecha_limite"] >= hoy:
                dias = (datetime.strptime(t["fecha_limite"], "%Y-%m-%d") - datetime.now()).days
                if dias <= 2:
                    proximas.append(f"⚠️ '{t['nombre']}' vence el {t['fecha_limite']} (en {dias} días)")
        if proximas:
            return "🔔 Recordatorios:\n" + "\n".join(proximas)
        return "🔔 No hay tareas próximas a vencer."

    def ayudar(self):
        return "\n".join(self.base_conocimiento.values())

    # --- Respuesta principal ---
    def responder(self, pregunta):
        intencion = self.clasificar_intencion(pregunta)
        if intencion == "agregar":
            return self.agregar_tarea(pregunta)
        elif intencion == "listar":
            return self.listar_tareas()
        elif intencion == "completar":
            return self.completar_tarea(pregunta)
        elif intencion == "eliminar":
            return self.eliminar_tarea(pregunta)
        elif intencion == "recordatorio":
            return self.recordatorios()
        else:
            return self.ayudar()

# --- EJEMPLO DE USO INTERACTIVO ---
if __name__ == "__main__":
    agente = AgenteTareasCurso()
    print("¡Bienvenido al Agente de Tareas del Curso! Escribe 'ayuda' para ver los comandos disponibles.")
    while True:
        comando_usuario = input("\nEscribe tu comando (o 'salir' para terminar): ")
        if comando_usuario.lower() in ["salir", "exit", "quit"]:
            print("¡Adiós!")
            break
        respuesta = agente.responder(comando_usuario)
        print(respuesta)


¡Bienvenido al Agente de Tareas del Curso! Escribe 'ayuda' para ver los comandos disponibles.

Escribe tu comando (o 'salir' para terminar): ayuda
Para agregar una tarea, dime: 'agregar tarea <nombre> para el <YYYY-MM-DD> con prioridad <alta/media/baja>'
Usa 'listar tareas' para ver todas tus tareas.
Usa 'completar tarea <nombre>' para marcar como hecha.
Usa 'eliminar tarea <nombre>' para borrarla.
Te recordaré las tareas próximas (vencen en menos de 2 días)

Escribe tu comando (o 'salir' para terminar): agregar tarea locura_total para el 2026-05-29 con prioridad alta
✅ Tarea 'tarea locura_total' agregada para el 2026-05-29 (prioridad 1).

Escribe tu comando (o 'salir' para terminar): listar
📋 Tareas pendientes:
1. 🟢 tarea locura_total - límite: 2026-05-29 (prioridad Alta)


Escribe tu comando (o 'salir' para terminar): salir
¡Adiós!
